In [1]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

In [2]:
class DrivingDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []

        for file in os.listdir(root_dir):
            if not file.endswith('.csv'):
                continue
            
            csv_path = os.path.join(root_dir, file)

            folder_name = file.replace('.csv', '')
            folder_path = os.path.join(root_dir, folder_name)
            
            if not os.path.isdir(folder_path):
                continue
            
            df = pd.read_csv(csv_path, header=None, sep=",")
            
            for _, row in df.iterrows():
                img_num = int(row[0])
                forward_steering = float(row[1])
                side_steering = float(row[2])
                
                img_filename = f"{img_num:04d}.jpg"
                img_path = os.path.join(folder_path, img_filename)
                
                if os.path.exists(img_path):
                    self.samples.append({
                        'image_path': img_path,
                        'forward_steering': forward_steering,
                        'side_steering': side_steering
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        image = Image.open(sample['image_path']).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        targets = torch.tensor([sample['forward_steering'], sample['side_steering']], dtype=torch.float32)
        
        return image, targets

In [3]:
data_transforms = transforms.Compose([
    transforms.ToTensor(),
])

driving_dataset = DrivingDataset(root_dir='dataset', transform=data_transforms)

print(f"Total valid samples found: {len(driving_dataset)}")

Total valid samples found: 7584


In [4]:
dataloader = DataLoader(
    driving_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=0,
    pin_memory=True
)

for images, targets in dataloader:
    print(f"Batch Image Tensor Shape: {images.shape}")
    print(f"Batch Target Tensor Shape: {targets.shape}")
    break

Batch Image Tensor Shape: torch.Size([32, 3, 224, 224])
Batch Target Tensor Shape: torch.Size([32, 2])


In [15]:
class DriverModel(torch.nn.Module):
    def __init__(self):
        super(DriverModel, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, kernel_size=5, stride=2)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=5, stride=2)
        self.conv3 = torch.nn.Conv2d(32, 64, kernel_size=5, stride=2)
        self.fc1 = torch.nn.LazyLinear(256)
        self.fc2 = torch.nn.Linear(256, 2)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = DriverModel().to(device)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
num_epochs = 30
for epoch in tqdm(range(num_epochs), desc="Training Epochs"):
    model.train()
    running_loss = 0.0
    
    for images, targets in dataloader:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
    
    epoch_loss = running_loss / len(driving_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")


Using device: cuda


Training Epochs:   3%|▎         | 1/30 [00:12<06:09, 12.75s/it]

Epoch [1/30], Loss: 0.2231


Training Epochs:   7%|▋         | 2/30 [00:25<05:50, 12.52s/it]

Epoch [2/30], Loss: 0.1179


...

Epoch [28/30], Loss: 0.0065


Training Epochs:  97%|█████████▋| 29/30 [06:01<00:12, 12.47s/it]

Epoch [29/30], Loss: 0.0061


Training Epochs: 100%|██████████| 30/30 [06:13<00:00, 12.46s/it]

Epoch [30/30], Loss: 0.0058


In [ ]:
import onnx

model.eval()

dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "driver_model.onnx",
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=["image_input"],
    output_names=["steering_output"],
    dynamic_axes={
        "image_input": {0: "batch_size"},
        "steering_output": {0: "batch_size"}
    },
    verbose=False
)

onnx_model = onnx.load("driver_model.onnx")
onnx.checker.check_model(onnx_model)

C:\Users\admin\AppData\Local\Temp\ipykernel_17196\2147597828.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0517 23:00:07.431000 17196 site-packages\torch\onnx\_internal\exporter\_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0517 23:00:08.011000 17196 site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_